In [1]:
!pip install -q langchain-groq langgraph pymupdf langchain_community langchain_chroma "unstructured[pdf]"
!apt install poppler-utils tesseract-ocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata
import os
from uuid import uuid4
from copy import deepcopy

from unstructured.documents.elements import NarrativeText, Title, Image, Table
import fitz
import re
from unstructured.partition.pdf import partition_pdf
from pypdf import PdfWriter, PdfReader


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b")
llm_summariser = ChatGroq(model="llama-3.1-8b-instant")
llm_image_summariser = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")


from langchain.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain.schema.document import Document
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings

In [3]:
def get_summary(text):
    messages = [SystemMessage("""
    Your job is to summarize the given raw text. Chunks of raw text are given to you and you have to analyze properly what all is contained in them.
    Pay special attention to table captions, image captions and formulas and include them in your summaries.
    You can skip the factual numeric details but you must tell what all is contained in the given chunk of raw text.

    IMPORTANT: Only output the summary and no boilerplate text like "Here's your summary"
    """)]
    messages.append(HumanMessage(f"Here is the raw text: {text}"))
    res = llm_summariser.invoke(messages)

    return res.content

def get_image_summary(image_data):
    messages = [SystemMessage("""
    Your job is to summarize the given image.
    If the image is graph, extract relevant details like graph title, axis labels, maximum values etc.
    If the image is a diagram or illustration extract important components and make proper analysis in them.
    Include keywords in your summary that will help map the summary easily to the image
    IMPORTANT: Only output the summary and no boilerplate text like "Here's your summary"
    """)]
    messages.append(HumanMessage(content=[
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
        }
    ]))
    res = llm_image_summariser.invoke(messages)

    return res.content

def get_table_summary(table_data):
    messages = [SystemMessage("""
    Your job is to summarize the given table. The table is in HTML format. Describe the table properly, include keywords that help in telling what all data is contained in the table.
    Make sure to include the names of column headings or row headings as necessary. Properly analyze the table and return a dense yet concise summary.
    IMPORTANT: Only output the summary and no boilerplate text like "Here's your summary"
    """)]
    messages.append(HumanMessage(f"Here is the table in HTML format: {table_data}"))
    res = llm_summariser.invoke(messages)

    return res.content

In [101]:
class Docstore():
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path


    def get_doc_hierarchy(self):

        def find_toc_pages(doc, search_limit=20):
            toc_pages = []
            for i in range(min(len(doc), search_limit)):
                page = doc[i]
                text = page.get_text("text", sort=True).lower()
                if "contents" in text or "table of contents" in text:
                    toc_pages.append(i)
            return toc_pages

        def toc_raw_to_hierarchy(toc_raw):
            messages = [SystemMessage("""
                You are a document hierarchy generator. Your task is to generate a python dictionary.
                You are given the raw text on the table of contents page of a document.
                analyze it properly and structure it into a neat json object.
                The document hierarchy is only needed till depth level 2. the schema of the json object is as follows
                {
                    "section_name": str
                    "section_children": list[str]
                }
                You must follow this schema.
                YOur output should be json object and it will be evaluated as it is so do not give boilerplate text or backticks
                """)]
            messages.append(HumanMessage("here is the raw text on the table of contents page of the document"))
            messages.append(HumanMessage(toc_raw))

            res = llm.with_structured_output(method="json_mode").invoke(messages)
            return res

        def toc_to_hierarchy(toc):
            hierarchy = []
            for i in range(len(toc)):
                if toc[i][0] == 1:
                    new_section = {
                        "section_name": toc[i][1],
                        "section_children": []
                    }
                    new_section_children = []
                    for j in range(i+1, len(toc)):
                        if toc[j][0] == 2:
                            new_section_children.append(toc[j][1])
                        elif toc[j][0] == 1:
                            break
                    new_section["section_children"] = new_section_children
                    hierarchy.append(new_section)
            return hierarchy

        doc = fitz.open(self.pdf_path)
        toc = doc.get_toc()
        toc_pages = find_toc_pages(doc)
        if toc:
            return toc_to_hierarchy(toc), toc_pages


        if toc_pages:
            toc_raw = ""
            for i in toc_pages:
                toc_raw += doc[i].get_text("text")
            return toc_raw_to_hierarchy(toc_raw), toc_pages

        return None

    def create_headings_store(self):

        self.low_res_elements = partition_pdf(
            filename="report.pdf",
            strategy="fast"
        )
        self.doc_hierarchy, self.toc_pages = self.get_doc_hierarchy()

        doc_hierarchy = deepcopy(self.doc_hierarchy)
        current_section_idx = 0
        current_section_chunks = []

        if self.toc_pages:
            last_toc_page = self.toc_pages[-1]
            split_idx = next(
                (i for i, x in enumerate(elements) if x.metadata.page_number == last_toc_page + 2),
                None
            )
            elements = elements[split_idx:]

        next_section_name = (
            doc_hierarchy[current_section_idx + 1]["section_name"]
            if current_section_idx + 1 < len(doc_hierarchy)
            else None
        )

        for e in self.low_res_elements:
            if next_section_name and isinstance(e, Title) and next_section_name in e.text:
                doc_hierarchy[current_section_idx]["section_chunks"] = current_section_chunks
                current_section_chunks = [e]
                current_section_idx += 1
                next_section_name = (
                    doc_hierarchy[current_section_idx + 1]["section_name"]
                    if current_section_idx + 1 < len(doc_hierarchy)
                    else None
                )
            else:
                current_section_chunks.append(e)

        doc_hierarchy[current_section_idx]["section_chunks"] = current_section_chunks


        leaf_sections = []
        for section in doc_hierarchy:
            if len(section["section_children"]) == 0:
                leaf_sections.append({
                    "section_id": str(uuid4()),
                    "section_name": section["section_name"],
                    "section_chunks": section["section_chunks"]
                })
            else:
                current_subsec_chunks = []
                split_idx = 0
                for i in range(len(section["section_chunks"])):
                    e = section["section_chunks"][i]
                    if isinstance(e, Title) and section["section_children"][0] in e.text:
                        leaf_sections.append({
                            "section_id": str(uuid4()),
                            "section_name": section["section_name"],
                            "section_chunks": current_subsec_chunks
                        })
                        split_idx = i
                        break
                    else:
                        current_subsec_chunks.append(e)

                section["section_chunks"] = section["section_chunks"][split_idx:]
                current_subsec_idx = 0
                current_subsec_chunks = []
                next_subsec_name = (
                    section["section_children"][current_subsec_idx + 1]
                    if current_subsec_idx + 1 < len(section["section_children"])
                    else None
                )
                for e in section["section_chunks"]:
                    if next_subsec_name and isinstance(e, Title) and next_subsec_name in e.text:
                        leaf_sections.append({
                            "section_id": str(uuid4()),
                            "section_name": section["section_children"][current_subsec_idx],
                            "section_chunks": current_subsec_chunks
                        })

                        current_subsec_chunks = [e]
                        current_subsec_idx += 1
                        next_subsec_name = (
                            section["section_children"][current_subsec_idx + 1]
                            if current_subsec_idx + 1 < len(section["section_children"])
                            else None
                        )
                    else:
                        current_subsec_chunks.append(e)

                leaf_sections.append({
                    "section_id": str(uuid4()),
                    "section_name": section["section_children"][current_subsec_idx],
                    "section_chunks": current_subsec_chunks
                })

        self.leaf_sections = leaf_sections

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500)

        summaries = {}

        for sec in leaf_sections:
            sec_text = []
            for sec_chunk in sec["section_chunks"]:
                if isinstance(sec_chunk, NarrativeText):
                    sec_text.append(sec_chunk.text)
            if len(sec_text) == 0:
                continue
            raw_sec_text = "\n".join(sec_text)
            sec_text_chunks = text_splitter.split_text(raw_sec_text)
            sec_summaries = []
            for sec_chunk in sec_text_chunks:
                sec_summaries.append(get_summary(sec_chunk))

            if len(sec_summaries) > 1:
                sec_summary = get_summary("\n".join(sec_summaries))
            else:
                sec_summary = sec_summaries[0]

            summaries[sec["section_id"]] = sec_summary

        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

        heading_store = Chroma(collection_name="heading_store", embedding_function=embeddings)

        # The storage layer for the parent documents
        store = InMemoryStore()
        id_key = "section_id"

        # The retriever (empty to start)
        heading_retriever = MultiVectorRetriever(
            vectorstore=heading_store,
            docstore=store,
            id_key=id_key,
        )
        heading_retriever.vectorstore.add_documents([
            Document(
                page_content=sec_sum,
                metadata={id_key: sec_id}
            )
            for sec_id, sec_sum in summaries.items()
        ])
        heading_retriever.docstore.mset([
            (sec["section_id"], sec)
            for sec in leaf_sections
        ])
        self.heading_retriever = heading_retriever
        self.page_cache = {}
        return heading_retriever

    def create_query_retriever(self, relevant_page_numbers):
        if not hasattr(self, "page_cache"):
            self.page_cache = {}

        pages_not_cached = []
        for page_number in relevant_page_numbers:
            if page_number not in self.page_cache:
                pages_not_cached.append(page_number)
        if len(pages_not_cached) != 0:
            writer = PdfWriter()
            input_pdf  = PdfReader("report.pdf")
            for i in pages_not_cached:
                writer.add_page(input_pdf.pages[i-1])

            batch_filename = 'elements.pdf'
            with open(batch_filename, 'wb') as output_file:
                writer.write(output_file)

            high_res_pages = partition_pdf(
                filename="elements.pdf",
                strategy="hi_res",
                extract_image_block_types=["Image"],
                extract_image_block_to_payload=True,
                infer_table_structure=True
            )

            newly_cached_pages = [[] for _ in pages_not_cached]
            for c in high_res_pages:
                newly_cached_pages[c.metadata.page_number - 1].append(c)

            for i in range(len(newly_cached_pages)):
                image_store = []
                table_store = []
                raw_text_store = []
                for c in newly_cached_pages[i]:
                    if isinstance(c, Image):
                        image_store.append({
                            "doc_id": str(uuid4()),
                            "doc_type": "image",
                            "image_data": c.metadata.image_base64,
                            "image_summary": get_image_summary(c.metadata.image_base64)
                        })
                    elif isinstance(c, Table):
                        table_store.append({
                            "doc_id": str(uuid4()),
                            "doc_type": "table",
                            "table_data": c.metadata.text_as_html,
                            "table_summary": get_table_summary(c.metadata.text_as_html)
                        })
                    else:
                        raw_text_store.append(c.text)


                    raw_text = "\n".join(raw_text_store)
                    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500)
                    raw_text_store = text_splitter.split_text(raw_text)
                    text_store = [{
                            "doc_id": str(uuid4()),
                            "doc_type": "text",
                            "text_data": t
                        } for t in raw_text_store]

                    new_cache = {
                        "high_res_chunks": newly_cached_pages[i],
                        "image_store": image_store,
                        "table_store": table_store,
                        "text_store": text_store
                    }
                self.page_cache[pages_not_cached[i]] = new_cache

        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        query_store = Chroma(collection_name="query_store", embedding_function=embeddings)
        query_doc_store = InMemoryStore()
        id_key="doc_id"

        query_retriever = MultiVectorRetriever(
            vectorstore=query_store,
            docstore=query_doc_store,
            id_key=id_key,
        )
        image_for_query = []
        table_for_query = []
        text_for_query = []

        for page in relevant_page_numbers:
            image_for_query += self.page_cache[page]["image_store"]
            table_for_query += self.page_cache[page]["table_store"]
            text_for_query += self.page_cache[page]["text_store"]

        if image_for_query:
            query_retriever.vectorstore.add_documents([
                Document(
                    page_content=image["image_summary"],
                    metadata={id_key: image["doc_id"], "doc_type": "image"}
                )
                for image in image_for_query
            ])
            query_retriever.docstore.mset([
                (image["doc_id"], image)
                for image in image_for_query
            ])

        if table_for_query:
            query_retriever.vectorstore.add_documents([
                Document(
                    page_content=table["table_summary"],
                    metadata={id_key: table["doc_id"], "doc_type": "table"}
                )
                for table in table_for_query
            ])
            query_retriever.docstore.mset([
                (table["doc_id"], table)
                for table in table_for_query
            ])

        if text_for_query:
            query_retriever.vectorstore.add_documents([
                Document(
                    page_content=text["text_data"],
                    metadata={id_key: text["doc_id"], "doc_type": "text"}
                )
                for text in text_for_query
            ])
            query_retriever.docstore.mset([
                (text["doc_id"], text)
                for text in text_for_query
            ])

        return query_retriever

### heading retriever test

In [3]:
def find_toc_pages(doc, search_limit=20):
    toc_pages = []
    for i in range(min(len(doc), search_limit)):
        page = doc[i]
        text = page.get_text("text", sort=True).lower()
        if "contents" in text or "table of contents" in text:
            toc_pages.append(i)
    return toc_pages

def toc_raw_to_hierarchy(toc_raw):
    messages = [SystemMessage("""
        You are a document hierarchy generator. Your task is to generate a python dictionary.
        You are given the raw text on the table of contents page of a document.
        analyze it properly and structure it into a neat json object.
        The document hierarchy is only needed till depth level 2. the schema of the json object is as follows
        {
            "section_name": str
            "section_children": list[str]
        }
        You must follow this schema.
        YOur output should be json object and it will be evaluated as it is so do not give boilerplate text or backticks
        """)]
    messages.append(HumanMessage("here is the raw text on the table of contents page of the document"))
    messages.append(HumanMessage(toc_raw))

    res = llm.with_structured_output(method="json_mode").invoke(messages)
    return res

def toc_to_hierarchy(toc):
    hierarchy = []
    for i in range(len(toc)):
        if toc[i][0] == 1:
            new_section = {
                "section_name": toc[i][1],
                "section_children": []
            }
            new_section_children = []
            for j in range(i+1, len(toc)):
                if toc[j][0] == 2:
                    new_section_children.append(toc[j][1])
                elif toc[j][0] == 1:
                    break
            new_section["section_children"] = new_section_children
            hierarchy.append(new_section)
    return hierarchy

def get_doc_hierarchy(pdf_path):
    doc = fitz.open(pdf_path)
    toc = doc.get_toc()
    toc_pages = find_toc_pages(doc)
    if toc:
        return toc_to_hierarchy(toc), toc_pages


    if toc_pages:
        toc_raw = ""
        for i in toc_pages:
            toc_raw += doc[i].get_text("text")
        return toc_raw_to_hierarchy(toc_raw), toc_pages

    return None


In [4]:
elements = partition_pdf(
    filename="report.pdf",
    strategy="fast"
    )

In [5]:
doc_hierarchy, toc_pages = get_doc_hierarchy("report.pdf")

In [6]:
current_section_idx = 0
current_section_chunks = []

if toc_pages:
    last_toc_page = toc_pages[-1]
    split_idx = next(
        (i for i, x in enumerate(elements) if x.metadata.page_number == last_toc_page + 2),
        None
    )
    elements = elements[split_idx:]

next_section_name = (
    doc_hierarchy[current_section_idx + 1]["section_name"]
    if current_section_idx + 1 < len(doc_hierarchy)
    else None
)

for e in elements:
    if next_section_name and isinstance(e, Title) and next_section_name in e.text:
        doc_hierarchy[current_section_idx]["section_chunks"] = current_section_chunks
        current_section_chunks = [e]
        current_section_idx += 1
        next_section_name = (
            doc_hierarchy[current_section_idx + 1]["section_name"]
            if current_section_idx + 1 < len(doc_hierarchy)
            else None
        )
    else:
        current_section_chunks.append(e)

doc_hierarchy[current_section_idx]["section_chunks"] = current_section_chunks

In [39]:
def get_level2_sections(doc_hierarchy):
    leaf_sections = []
    for section in doc_hierarchy:
        if len(section["section_children"]) == 0:
            leaf_sections.append({
                "section_id": str(uuid4()),
                "section_name": section["section_name"],
                "section_chunks": section["section_chunks"]
            })
        else:
            current_subsec_chunks = []
            split_idx = 0
            for i in range(len(section["section_chunks"])):
                e = section["section_chunks"][i]
                if isinstance(e, Title) and section["section_children"][0] in e.text:
                    leaf_sections.append({
                        "section_id": str(uuid4()),
                        "section_name": section["section_name"],
                        "section_chunks": current_subsec_chunks
                    })
                    split_idx = i
                    break
                else:
                    current_subsec_chunks.append(e)

            section["section_chunks"] = section["section_chunks"][split_idx:]
            current_subsec_idx = 0
            current_subsec_chunks = []
            next_subsec_name = (
                section["section_children"][current_subsec_idx + 1]
                if current_subsec_idx + 1 < len(section["section_children"])
                else None
            )
            for e in section["section_chunks"]:
                if next_subsec_name and isinstance(e, Title) and next_subsec_name in e.text:
                    leaf_sections.append({
                        "section_id": str(uuid4()),
                        "section_name": section["section_children"][current_subsec_idx],
                        "section_chunks": current_subsec_chunks
                    })

                    current_subsec_chunks = [e]
                    current_subsec_idx += 1
                    next_subsec_name = (
                        section["section_children"][current_subsec_idx + 1]
                        if current_subsec_idx + 1 < len(section["section_children"])
                        else None
                    )
                else:
                    current_subsec_chunks.append(e)

            leaf_sections.append({
                "section_id": str(uuid4()),
                "section_name": section["section_children"][current_subsec_idx],
                "section_chunks": current_subsec_chunks
            })

    return leaf_sections

In [40]:
leaf_sections = get_level2_sections(doc_hierarchy)

In [ ]:
leaf_sections

In [27]:
def get_summary(text):
    messages = [SystemMessage("""
    Your job is to summarize the given raw text. Chunks of raw text are given to you and you have to analyze properly what all is contained in them.
    Pay special attention to table captions, image captions and formulas and include them in your summaries.
    You can skip the factual numeric details but you must tell what all is contained in the given chunk of raw text.

    IMPORTANT: Only output the summary and no boilerplate text like "Here's your summary"
    """)]
    messages.append(HumanMessage(f"Here is the raw text: {text}"))
    res = llm_summariser.invoke(messages)

    return res.content

In [42]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500)

summaries = {}

for sec in leaf_sections:
    sec_text = []
    for sec_chunk in sec["section_chunks"]:
        if isinstance(sec_chunk, NarrativeText):
            sec_text.append(sec_chunk.text)
    if len(sec_text) == 0:
        continue
    raw_sec_text = "\n".join(sec_text)
    sec_text_chunks = text_splitter.split_text(raw_sec_text)
    sec_summaries = []
    for sec_chunk in sec_text_chunks:
        sec_summaries.append(get_summary(sec_chunk))

    if len(sec_summaries) > 1:
        sec_summary = get_summary("\n".join(sec_summaries))
    else:
        sec_summary = sec_summaries[0]

    summaries[sec["section_id"]] = sec_summary

In [44]:


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

heading_store = Chroma(collection_name="heading_store", embedding_function=embeddings)

# The storage layer for the parent documents
store = InMemoryStore()
id_key = "section_id"

# The retriever (empty to start)
heading_retriever = MultiVectorRetriever(
    vectorstore=heading_store,
    docstore=store,
    id_key=id_key,
)

In [50]:
heading_retriever.vectorstore.add_documents([
    Document(
        page_content=sec_sum,
        metadata={id_key: sec_id}
    )
    for sec_id, sec_sum in summaries.items()
])
heading_retriever.docstore.mset([
    (sec["section_id"], sec)
    for sec in leaf_sections
])

In [67]:
docs = heading_retriever.invoke("What is the per layer complexity?")

### section cache test

In [5]:
docstore = Docstore("report.pdf")
heading_retriever = docstore.create_headings_store()

/tmp/ipython-input-2898470464.py:193: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipython-input-2898470464.py:195: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  heading_store = Chroma(collection_name="heading_store", embedding_function=embeddings)


In [6]:
docs = heading_retriever.invoke("What is the per layer complexity?")

In [ ]:
# {
#     "section_id": {
#         "high_res_chunks": any,
#         "image_store": [{
#             "image_id": str,
#             "image_data": str,
#             "image_summary": str
#         },...],
#         "table_store": [{
#             "table_id": str,
#             "table_data": str,
#             "table_summary": str
#         },...],
#         "text_store": [{
#             "text_id": str,
#             "text_data": str,
#         },...],

#     },...
# }

In [9]:
page_cache = {}

In [72]:
# use sections in docs and section cache to create vector index of reqd sections.
# retrieve docs from that vector index
# send docs to evaluator llm, check if new vector index need or new docs from old index needed

In [10]:
query = "What is the per layer complexity?"

# improve this section, add re-ranking, query rewriting etc.
docs = heading_retriever.invoke(query)

# retrieve page numbers of relevant sections
relevant_page_numbers = set()
for doc in docs:
    for c in doc["section_chunks"]:
        relevant_page_numbers.add(c.metadata.page_number)

In [11]:
pages_not_cached = []
for page_number in relevant_page_numbers:
    if page_number not in page_cache:
        pages_not_cached.append(page_number)

In [12]:
from pypdf import PdfWriter, PdfReader

In [13]:
writer = PdfWriter()
input_pdf  = PdfReader("report.pdf")
for i in pages_not_cached:
    writer.add_page(input_pdf.pages[i-1])

batch_filename = 'elements.pdf'
with open(batch_filename, 'wb') as output_file:
    writer.write(output_file)

In [28]:
high_res_pages = partition_pdf(
    filename="elements.pdf",
    strategy="hi_res",
    extract_image_block_types=["Image"],
    extract_image_block_to_payload=True,
    infer_table_structure=True
)

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

In [16]:
import base64
# open image.jpeg and store its base64 data in variable image
with open("image.jpeg", "rb") as f:
    image = base64.b64encode(f.read()).decode("utf-8")

In [30]:
llm_image_summariser = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

def get_image_summary(image_data):
    messages = [SystemMessage("""
    Your job is to summarize the given image.
    If the image is graph, extract relevant details like graph title, axis labels, maximum values etc.
    If the image is a diagram or illustration extract important components and make proper analysis in them.
    Include keywords in your summary that will help map the summary easily to the image
    IMPORTANT: Only output the summary and no boilerplate text like "Here's your summary"
    """)]
    messages.append(HumanMessage(content=[
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
        }
    ]))
    res = llm_image_summariser.invoke(messages)

    return res.content

def get_table_summary(table_data):
    messages = [SystemMessage("""
    Your job is to summarize the given table. The table is in HTML format. Describe the table properly, include keywords that help in telling what all data is contained in the table.
    Make sure to include the names of column headings or row headings as necessary. Properly analyze the table and return a dense yet concise summary.
    IMPORTANT: Only output the summary and no boilerplate text like "Here's your summary"
    """)]
    messages.append(HumanMessage(f"Here is the table in HTML format: {table_data}"))
    res = llm_summariser.invoke(messages)

    return res.content

In [ ]:
newly_cached_pages = [[] for _ in pages_not_cached]
for c in high_res_pages:
    newly_cached_pages[c.metadata.page_number - 1].append(c)

for i in range(len(newly_cached_pages)):
    image_store = []
    table_store = []
    raw_text_store = []
    for c in newly_cached_pages[i]:
        if isinstance(c, Image):
            image_store.append({
                "doc_id": str(uuid4()),
                "doc_type": "image",
                "image_data": c.metadata.image_base64,
                "image_summary": get_image_summary(c.metadata.image_base64)
            })
        elif isinstance(c, Table):
            table_store.append({
                "doc_id": str(uuid4()),
                "doc_type": "table",
                "table_data": c.metadata.text_as_html,
                "table_summary": get_table_summary(c.metadata.text_as_html)
            })
        else:
            raw_text_store.append(c.text)


        raw_text = "\n".join(raw_text_store)
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500)
        raw_text_store = text_splitter.split_text(raw_text)
        text_store = [{
                "doc_id": str(uuid4()),
                "doc_type": "text",
                "text_data": t
            } for t in raw_text_store]

        new_cache = {
            "high_res_chunks": newly_cached_pages[i],
            "image_store": image_store,
            "table_store": table_store,
            "text_store": text_store
        }
    page_cache[pages_not_cached[i]] = new_cache

In [50]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

query_store = Chroma(collection_name="query_store", embedding_function=embeddings)

# The storage layer for the parent documents
query_doc_store = InMemoryStore()
id_key="doc_id"

# The retriever (empty to start)
query_retriever = MultiVectorRetriever(
    vectorstore=query_store,
    docstore=query_doc_store,
    id_key=id_key,
)
image_for_query = []
table_for_query = []
text_for_query = []

for page in relevant_page_numbers:
    image_for_query += page_cache[page]["image_store"]
    table_for_query += page_cache[page]["table_store"]
    text_for_query += page_cache[page]["text_store"]

query_retriever.vectorstore.add_documents([
    Document(
        page_content=image["image_summary"],
        metadata={id_key: image["doc_id"], "doc_type": "image"}
    )
    for image in image_for_query
])
query_retriever.docstore.mset([
    (image["doc_id"], image)
    for image in image_for_query
])

query_retriever.vectorstore.add_documents([
    Document(
        page_content=table["table_summary"],
        metadata={id_key: table["doc_id"], "doc_type": "table"}
    )
    for table in table_for_query
])
query_retriever.docstore.mset([
    (table["doc_id"], table)
    for table in table_for_query
])

query_retriever.vectorstore.add_documents([
    Document(
        page_content=text["text_data"],
        metadata={id_key: text["doc_id"], "doc_type": "text"}
    )
    for text in text_for_query
])
query_retriever.docstore.mset([
    (text["doc_id"], text)
    for text in text_for_query
])

In [53]:
docs = query_retriever.invoke(query)

In [55]:
docs[0]

{'doc_id': 'f8fde5eb-4bb5-44d0-84b5-6cedf7c2ac2c',
 'doc_type': 'table',
 'table_data': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Attention</td><td>O(n? - d)</td><td>O(1)</td><td>O(1)</td></tr><tr><td>Recurrent</td><td>O(n - d?)</td><td>O(n)</td><td>O(n)</td></tr><tr><td>Convolutional</td><td>O(k-n-d?)</td><td>O(1)</td><td>O(logx(n))</td></tr><tr><td>Self-Attention (restricted)</td><td>O(r-n-d)</td><td>ol)</td><td>O(n/r)</td></tr></tbody></table>',
 'table_summary': "The table is a comparison of different neural network layer types, specifically in terms of their computational complexity and scalability characteristics. \n\nIt includes four types of layers: Self-Attention, Recurrent, Convolutional, and Self-Attention (restricted). The table provides information on the complexity per layer, sequential operations, and maximum path length for each type.\n\nThe complexity per

## test

In [74]:
docstore = Docstore("report.pdf")
heading_retriever = docstore.create_headings_store()

In [90]:
query = "what is the scaling factor in scaled dot product attention"

# improve this section, add re-ranking, query rewriting etc.
docs = heading_retriever.invoke(query)

In [92]:
relevant_page_numbers = set()
for doc in docs:
    for c in doc["section_chunks"]:
        relevant_page_numbers.add(c.metadata.page_number)

In [93]:
relevant_page_numbers

{3, 4, 5, 8, 9}

In [94]:
query_retriever = docstore.create_query_retriever(relevant_page_numbers)

In [95]:
final_docs = query_retriever.invoke(query)

In [ ]:
final_docs

In [97]:
query = "How many identical layers are present in the decoder?"

# improve this section, add re-ranking, query rewriting etc.
docs = heading_retriever.invoke(query)

# retrieve page numbers of relevant sections
relevant_page_numbers = set()
for doc in docs:
    for c in doc["section_chunks"]:
        relevant_page_numbers.add(c.metadata.page_number)

In [104]:
relevant_page_numbers

{3}

In [105]:
query_retriever_2 = docstore.create_query_retriever(relevant_page_numbers)

In [106]:
final_docs = query_retriever_2.invoke(query)

In [108]:
final_docs

[{'doc_id': '93893ada-7ea4-4cc9-9218-917f380c6f09',
  'doc_type': 'text',
  'text_data': 'Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimensi